# Nephron - Stage 2 model (Enhanced Risk Screening) trained on CKD_400k.csv

This notebook follows CRISP-DM. It trains **Random Forest** and **XGBoost** on the Enhanced Risk Screening inputs (27 Stage 1
answers + blood pressure + 18 laboratory values + eGFR = **48 inputs**) of **CKD_400k.csv**, the same dataset as the Stage 1 model,
selects one with a rule fixed before training, isotonic-calibrates it, sets the threshold, evaluates it **once** on the held-out
test rows, and saves it in exactly the form the website checks (`ml/model_stage2/`).

It replaces the earlier Stage 2 model, which had been trained on a different dataset (CKD_RiskAssessment_Synthetic.csv). Only the
**dataset** changes: the inputs, the preprocessing, the model-selection rule and the threshold rule are the same as before, so the
website needs no code change.

## How to use it
1. Use the **same Python environment as the Stage 1 notebook** - the versions pinned in `requirements.txt`
   (scikit-learn 1.8.0, xgboost 3.1.3, pandas 3.0.2, numpy 2.4.4). The website refuses a model saved with other versions.
2. Check `PROJECT_DIR` and `DATA_PATH` in the **Settings** cell.
3. **Kernel > Restart and Run All.** A full run takes a long time (roughly 1.5 to 3 hours on a laptop - the Random Forest steps
   are the slowest). To check that everything runs first, set `SAMPLE_ROWS_FOR_A_QUICK_TRIAL = 40000` (the result is labelled
   QUICKTRIAL and must not be used), then set it back to `None` and run again.
4. In a terminal in the project folder: `python verify_model_stage2.py` (every line PASS), then
   `python model_report.py --stage 2 --csv ml\stage2_sealed_test.csv --target CKD` (the administrator's Model page figures).

## Splits - the same people as Stage 1
The data are split exactly like the Stage 1 notebook (stratified 70 / 15 / 15, random_state 42). The **60,000 test rows are the
same people** as Stage 1's test rows, so both models are evaluated on the same held-out people. Stage 2 has no separate
threshold-validation step (its threshold comes from out-of-fold predictions), so it trains on the other **340,000** rows
(Stage 1's training + validation rows).

## Read this before you present the results
* **The data is synthetic.** Results describe data like CKD_400k.csv, not real patients.
* **Near-circular by construction.** eGFR, serum creatinine and UACR are part of how CKD is defined; in CKD_400k.csv the plain rule
  `eGFR < 60 or UACR > 300` already agrees with the CKD label on about 99% of rows. Near-perfect scores are therefore expected and
  are not evidence of predictive power - the notebook reports the plain rule and a block-by-block (tier) analysis next to the model.

## 1. Business understanding
* **Goal:** a confirmation-level **screening** estimate built on a person's preliminary screening, once laboratory results exist. Never a diagnosis.
* **Inputs:** 48 (the same as the website's Enhanced Screening). eGFR is not asked: the website computes it with CKD-EPI 2021 from serum
  creatinine, age and sex; this notebook verifies the dataset's eGFR is exactly that value.
* **Only serum creatinine is a required laboratory value;** blank optional values are imputed by the saved pipeline (missing-value indicators for
  the laboratory values that have blanks in the training data).
* **Models:** Random Forest and XGBoost only. Threshold: the highest calibrated probability at which pooled out-of-fold sensitivity is still >= 0.98.

## 2. Settings and environment

In [8]:
# ======================= SETTINGS: the only cell you may need to edit =======================
from pathlib import Path

PROJECT_DIR =  Path(r"D:\4ADA\NEPHRON_DEVELOPMENT\Nephron")   # the Flask project (it contains app.py and the ml folder)
DATA_PATH = Path(r"D:\4ADA\NEPHRON_DEVELOPMENT\Nephron\CKD_400k.csv")                     # e.g. Path(r"D:\...\CKD_400k.csv"). None = look for CKD_400k.csv near this notebook and the project
MODEL_DIR = PROJECT_DIR / "ml" / "model_stage2"   # where the website looks for the Stage 2 model. Do not change it.
FILE_NAME = "final_model_stage2.pkl"

TARGET = "CKD"
HIGH_RISK_CLASS = 1
RANDOM_STATE = 42
SAMPLE_ROWS_FOR_A_QUICK_TRIAL = None # e.g. 40000 to test the notebook quickly. Leave None for the real run.
TUNING_ROWS = 100_000                # rows used to search hyper-parameters (the final model and OOF use ALL training rows)
CV_FOLDS = 5
N_ITER_RF, N_ITER_XGB = 20, 40       # candidates tried by each search
TARGET_RECALL = 0.98                 # the threshold rule (unchanged from the earlier Stage 2 model)
TIE_MARGIN_PRECISION, TIE_MARGIN_PRAUC = 0.005, 0.002
PREFERRED_MODEL_ON_TIE = "XGBoost"   # one algorithm family for both stages when the data cannot tell them apart
N_JOBS = -1

In [9]:
import datetime as dt
import hashlib
import json
import sys
import time
import warnings
from importlib.metadata import PackageNotFoundError, version

import joblib
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import loguniform, randint, uniform

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)

try:
    import xgboost
except ImportError:
    raise SystemExit("XGBoost is not installed in this environment. Run:  pip install xgboost-cpu==3.1.3  then restart the kernel.")
xgboost_version = None
for dist in ("xgboost", "xgboost-cpu"):          # the website's requirements use the CPU-only build, which imports as xgboost
    try:
        xgboost_version = version(dist)
        break
    except PackageNotFoundError:
        pass
xgboost_version = xgboost_version or xgboost.__version__

installed = {"scikit-learn": sklearn.__version__, "xgboost": xgboost_version, "pandas": pd.__version__, "numpy": np.__version__}
pins = {}
for line in (PROJECT_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    for name in ("scikit-learn", "xgboost-cpu", "xgboost", "pandas", "numpy"):
        if line.startswith(name + "=="):
            pins["xgboost" if name.startswith("xgboost") else name] = line.split("==")[1].strip()
print("Python", sys.version.split()[0])
for name, ver in installed.items():
    print(f"{name:13s} installed {ver:10s} requirements.txt {pins.get(name, '-')}")
different = {n: (installed[n], pins[n]) for n in pins if installed[n] != pins[n]}
if different and not SAMPLE_ROWS_FOR_A_QUICK_TRIAL:
    raise SystemExit("These versions differ from requirements.txt, so the website would refuse the model: " + str(different)
                     + "\nInstall the pinned versions (pip install -r requirements.txt) and restart the kernel.")
assert (PROJECT_DIR / "ml" / "lab_catalog.json").exists(), f"{PROJECT_DIR} is not the Flask project. Fix PROJECT_DIR in the Settings cell."

Python 3.13.5
scikit-learn  installed 1.8.0      requirements.txt 1.8.0
xgboost       installed 3.1.3      requirements.txt 3.1.3
pandas        installed 3.0.2      requirements.txt 3.0.2
numpy         installed 2.4.4      requirements.txt 2.4.4


## 3. Model-selection protocol - fixed before any model is trained
1. **Candidates:** Random Forest and XGBoost, both tuned (RandomizedSearchCV, stratified 5-fold, PR-AUC) on a sample of the training rows.
2. **No class weights or resampling** - the trade-off is handled by the decision threshold.
3. **Candidate evaluation:** out-of-fold (OOF) predictions of each tuned candidate on ALL training rows, the same 5 folds, with
   **cross-fitted isotonic calibration** (each fold calibrated by a curve that never saw it).
4. **Operating point:** the highest calibrated threshold at which pooled OOF sensitivity >= `TARGET_RECALL` (0.98).
5. **Selection:** higher mean OOF precision at that operating point; a tie (difference < 0.005) -> higher mean OOF PR-AUC; a tie (< 0.002)
   -> `PREFERRED_MODEL_ON_TIE` (XGBoost). The website's per-result explanations need XGBoost, so if Random Forest wins clearly the notebook
   stops before saving and says so - that is a decision for the team.
6. The **test rows are not used** until section 9, once.

## 4. Data understanding

In [10]:
def find_data_file():
    if DATA_PATH is not None:
        return Path(DATA_PATH)
    here = Path.cwd()
    for folder in (here, here.parent, here.parent.parent, PROJECT_DIR, PROJECT_DIR.parent, PROJECT_DIR.parent / "NEPHRON_ML"):
        if (folder / "CKD_400k.csv").exists():
            return folder / "CKD_400k.csv"
    raise FileNotFoundError("CKD_400k.csv was not found. Set DATA_PATH in the Settings cell.")

data_file = find_data_file()
raw = pd.read_csv(data_file, keep_default_na=False, na_values=[""])      # only an empty cell is missing (as in Stage 1)
data_sha256 = hashlib.sha256(data_file.read_bytes()).hexdigest()
print(f"{data_file} | {len(raw):,} rows | sha256 {data_sha256[:16]}...")
assert set(raw[TARGET].unique()) == {0, 1}
print(f"CKD rate: {raw[TARGET].mean():.1%}")

D:\4ADA\NEPHRON_DEVELOPMENT\Nephron\CKD_400k.csv | 400,000 rows | sha256 5766a17adfd21104...
CKD rate: 49.3%


In [11]:
# The 48 inputs: the website's own contract (the same names, in the same order, as the Stage 2 model it replaces).
lab_catalog = json.loads((PROJECT_DIR / "ml" / "lab_catalog.json").read_text(encoding="utf-8"))
LAB_FIELDS = [f["name"] for f in lab_catalog["fields"]]                     # 20: blood pressure + 18 laboratory values
STAGE1_FIELDS = ["Country", "Sex", "Ethnicity", "Age", "BMI", "Smoking_Status", "Alcohol_Consumption", "Physical_Activity_Level",
                 "Exercise_Hours_Per_Week", "Water_Intake_L", "Fast_Food_Frequency_Per_Week", "Sleep_Duration_Hours", "Diabetes",
                 "Cardiovascular_Disease", "Hyperlipidemia", "Recurrent_UTI", "Family_History_CKD", "Hypertension", "Heart_Failure",
                 "Kidney_Stones", "Autoimmune_Disease", "ACE_Inhibitor", "Diabetes_Medication", "Diuretic", "ARB", "Statin", "NSAID_Usage"]
INPUT_COLUMNS = STAGE1_FIELDS + LAB_FIELDS[:3] + ["eGFR"] + LAB_FIELDS[3:]   # ... Systolic_BP, Diastolic_BP, Serum_Creatinine, eGFR, BUN ...
assert len(INPUT_COLUMNS) == 48 and len(set(INPUT_COLUMNS)) == 48
existing = MODEL_DIR / "model_metadata.json"
if existing.exists():                                                       # the contract the website already uses
    assert json.loads(existing.read_text(encoding="utf-8"))["features"] == INPUT_COLUMNS, "input order differs from the current Stage 2 model"
missing_columns = [c for c in INPUT_COLUMNS if c not in raw.columns]
assert not missing_columns, f"not in the dataset: {missing_columns}"

CATEGORICAL = ["Country", "Sex", "Ethnicity", "Smoking_Status", "Physical_Activity_Level"]
CATEGORICAL_BLANK_IS_CATEGORY = ["Alcohol_Consumption"]                    # blank = its own category, Not_Reported
BINARY = [c for c in STAGE1_FIELDS if c not in CATEGORICAL + CATEGORICAL_BLANK_IS_CATEGORY + ["Age", "BMI", "Exercise_Hours_Per_Week",
          "Water_Intake_L", "Fast_Food_Frequency_Per_Week", "Sleep_Duration_Hours"]]
NUMERIC = [c for c in INPUT_COLUMNS if c not in CATEGORICAL + CATEGORICAL_BLANK_IS_CATEGORY + BINARY]
MISSING_LABEL = "Not_Reported"
REQUIRED = ["Age", "Sex", "BMI", "Serum_Creatinine"]
RENAL_MARKERS = ["eGFR", "Serum_Creatinine", "Urine_ACR"]
print(f"{len(NUMERIC)} numeric, {len(BINARY)} yes/no, {len(CATEGORICAL) + 1} categorical")
print("Blank share per input (only blanks the website may also leave blank are allowed):")
blank = raw[INPUT_COLUMNS].isna().mean()
print(blank[blank > 0].round(3).to_string())
assert not raw[REQUIRED].isna().any().any(), "a required input has blanks"
assert set(blank[blank > 0].index) <= set(LAB_FIELDS + CATEGORICAL_BLANK_IS_CATEGORY), "an unexpected input has blanks"

27 numeric, 15 yes/no, 6 categorical
Blank share per input (only blanks the website may also leave blank are allowed):
Alcohol_Consumption    0.363
Urine_ACR              0.132
Urine_Protein          0.132
HbA1c                  0.177
Fasting_Glucose        0.062
Calcium                0.040
Phosphorus             0.288
HDL                    0.029
LDL                    0.030
CRP                    0.042


In [12]:
# eGFR must be exactly what the website computes (CKD-EPI 2021, rounded to 1 decimal), or the model learns from values it never receives.
def ckd_epi_2021(scr, age, sex):
    female = (sex == "Female").to_numpy()
    k, a = np.where(female, 0.7, 0.9), np.where(female, -0.241, -0.302)
    ratio = scr.to_numpy() / k
    value = 142 * np.minimum(ratio, 1) ** a * np.maximum(ratio, 1) ** -1.200 * 0.9938 ** age.to_numpy() * np.where(female, 1.012, 1.0)
    return np.round(value, 1)

same = np.abs(ckd_epi_2021(raw["Serum_Creatinine"], raw["Age"], raw["Sex"]) - raw["eGFR"].to_numpy()) <= 0.05
print(f"Dataset eGFR == CKD-EPI 2021 on {same.mean():.4%} of rows")
assert same.mean() > 0.999
assert (raw["Obesity"] == (raw["BMI"] >= 30).astype(int)).all(), "Obesity is not BMI >= 30 in this data"

rule_all = ((raw["eGFR"] < 60) | (raw["Urine_ACR"] > 300)).astype(int)
print(f"Plain rule 'eGFR < 60 or UACR > 300' agrees with the CKD label on {(rule_all == raw[TARGET]).mean():.2%} of rows (near-circularity)")
for name in CATEGORICAL + CATEGORICAL_BLANK_IS_CATEGORY:
    print(f"{name}: {sorted(raw[name].dropna().astype(str).unique())}")

Dataset eGFR == CKD-EPI 2021 on 100.0000% of rows
Plain rule 'eGFR < 60 or UACR > 300' agrees with the CKD label on 99.14% of rows (near-circularity)
Country: ['Australia', 'Canada', 'India', 'Other', 'UK', 'USA']
Sex: ['Female', 'Male']
Ethnicity: ['Asian', 'Black', 'Hispanic', 'Other', 'White']
Smoking_Status: ['Current', 'Former', 'Never']
Physical_Activity_Level: ['Active', 'Light', 'Moderate', 'Sedentary']
Alcohol_Consumption: ['Heavy', 'Moderate']


## 5. Data preparation: the same split as Stage 1, and the preprocessing pipeline

In [13]:
from sklearn.model_selection import StratifiedKFold, train_test_split

data = raw.sample(SAMPLE_ROWS_FOR_A_QUICK_TRIAL, random_state=RANDOM_STATE) if SAMPLE_ROWS_FOR_A_QUICK_TRIAL else raw
if SAMPLE_ROWS_FOR_A_QUICK_TRIAL:
    print(f"QUICK TRIAL on {len(data):,} rows - NOT the real model.\n")
X, y = data[INPUT_COLUMNS].copy(), data[TARGET].astype(int)
# Exactly the Stage 1 notebook's two splits (70% / 30%, then the 30% halved) -> the same 60,000 test rows.
X_s1_train, X_hold, y_s1_train, y_hold = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_s1_val, X_test_sealed, y_s1_val, y_test_sealed = train_test_split(X_hold, y_hold, test_size=0.50, stratify=y_hold, random_state=RANDOM_STATE)
X_train = pd.concat([X_s1_train, X_s1_val])
y_train = pd.concat([y_s1_train, y_s1_val])
print(f"training {len(X_train):,} rows (CKD {y_train.mean():.3f}) | sealed test {len(X_test_sealed):,} rows (CKD {y_test_sealed.mean():.3f})")
print("The sealed test rows are not used again until section 9.")

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
splits = list(cv.split(X_train, y_train))          # the same folds for OOF evaluation, calibration and the final fit
fold_id = np.zeros(len(y_train), dtype=int)
for k, (_, va) in enumerate(splits):
    fold_id[va] = k
y_tr = y_train.to_numpy()

training 340,000 rows (CKD 0.493) | sealed test 60,000 rows (CKD 0.493)
The sealed test rows are not used again until section 9.


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Binarizer, OneHotEncoder

OBESITY_THRESHOLD = 30 - 0.05           # BMI has 1 decimal: 29.9 -> 0, 30.0 -> 1

def make_preprocessor(columns=None):
    """A NEW, unfitted preprocessor (the same design as the earlier Stage 2 model, so the website reads it the same way)."""
    columns = columns or INPUT_COLUMNS
    keep = lambda names: [c for c in names if c in columns]
    one_hot = lambda: OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    parts = [("numeric", SimpleImputer(strategy="median", add_indicator=True), keep(NUMERIC)),
             ("binary", SimpleImputer(strategy="most_frequent"), keep(BINARY)),
             ("derived_obesity", Pipeline([("impute", SimpleImputer(strategy="median")),
                                           ("threshold", Binarizer(threshold=OBESITY_THRESHOLD))]), ["BMI"]),
             ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", one_hot())]), keep(CATEGORICAL)),
             ("categorical_blank_is_category", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=MISSING_LABEL)),
                                                          ("onehot", one_hot())]), keep(CATEGORICAL_BLANK_IS_CATEGORY))]
    return ColumnTransformer([p for p in parts if p[2]], sparse_threshold=0.0)

def make_pipeline(model, columns=None):
    return Pipeline([("preprocessor", make_preprocessor(columns)), ("model", model)])

probe = make_preprocessor().fit(X_train)
names = list(probe.get_feature_names_out())
print(f"{len(INPUT_COLUMNS)} inputs -> {len(names)} model features")
print("Missing-value indicators for:", [n.split("missingindicator_")[1] for n in names if "missingindicator_" in n])

48 inputs -> 75 model features
Missing-value indicators for: ['Urine_ACR', 'Urine_Protein', 'HbA1c', 'Fasting_Glucose', 'Calcium', 'Phosphorus', 'HDL', 'LDL', 'CRP']


## 6. Modeling: tune both candidates

In [15]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

def make_rf():
    return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS)

def make_xgb():
    return XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist", importance_type="gain",
                         n_jobs=N_JOBS, random_state=RANDOM_STATE)

rf_space = {"model__n_estimators": randint(100, 301), "model__max_depth": randint(6, 17), "model__min_samples_leaf": randint(10, 151),
            "model__max_features": ["sqrt", 0.3, 0.5], "model__max_samples": [0.5, 0.7, None]}
xgb_space = {"model__n_estimators": randint(150, 701), "model__max_depth": randint(3, 9), "model__learning_rate": loguniform(0.01, 0.2),
             "model__min_child_weight": randint(1, 21), "model__subsample": uniform(0.6, 0.4), "model__colsample_bytree": uniform(0.5, 0.5),
             "model__gamma": uniform(0.0, 1.0), "model__reg_alpha": loguniform(1e-3, 10), "model__reg_lambda": loguniform(0.1, 10)}

tune_rows = min(TUNING_ROWS, len(X_train))
X_tune = X_train.sample(tune_rows, random_state=RANDOM_STATE)
y_tune = y_train.loc[X_tune.index]
tune_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def run_search(name, model, space, n_iter):
    search = RandomizedSearchCV(make_pipeline(model), space, n_iter=n_iter, cv=tune_cv, random_state=RANDOM_STATE, n_jobs=1,
                                scoring={"pr_auc": "average_precision", "roc_auc": "roc_auc", "neg_brier": "neg_brier_score"},
                                refit="pr_auc", error_score="raise")
    t0 = time.perf_counter(); search.fit(X_tune, y_tune); seconds = time.perf_counter() - t0
    i = search.best_index_
    best = {k.replace("model__", ""): (v.item() if hasattr(v, "item") else v) for k, v in search.best_params_.items()}
    cvres = {m: float(search.cv_results_[f"mean_test_{m}"][i]) for m in ("pr_auc", "roc_auc", "neg_brier")}
    print(f"{name}: best CV PR-AUC {cvres['pr_auc']:.4f} on {tune_rows:,} rows ({seconds/60:.1f} min) | {best}")
    return search, {"best_params": best, "tuning_cv": cvres, "n_candidates": n_iter, "tuning_rows": tune_rows, "search_seconds": round(seconds, 1)}

rf_search, rf_summary = run_search("Random Forest", make_rf(), rf_space, N_ITER_RF)
xgb_search, xgb_summary = run_search("XGBoost", make_xgb(), xgb_space, N_ITER_XGB)

Random Forest: best CV PR-AUC 0.9992 on 100,000 rows (17.9 min) | {'max_depth': 10, 'max_features': 0.3, 'max_samples': None, 'min_samples_leaf': 17, 'n_estimators': 274}
XGBoost: best CV PR-AUC 0.9993 on 100,000 rows (5.9 min) | {'colsample_bytree': 0.662172510502637, 'gamma': 0.12208795470067335, 'learning_rate': 0.02907730652732417, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 534, 'reg_alpha': 0.008160948743089917, 'reg_lambda': 0.7148510793512985, 'subsample': 0.9272059063689972}


## 7. Out-of-fold evaluation with cross-fitted isotonic calibration, and the threshold of each candidate

In [16]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import cross_val_predict

def crossfit_isotonic(p_raw):
    p_cal = np.zeros_like(p_raw)
    for k in range(CV_FOLDS):
        fit, apply = fold_id != k, fold_id == k
        p_cal[apply] = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(p_raw[fit], y_tr[fit]).predict(p_raw[apply])
    return p_cal

def threshold_for_recall(y_true, p, target):
    """Highest threshold whose sensitivity is still >= target."""
    _, recall, thresholds = precision_recall_curve(y_true, p)
    return float(thresholds[np.where(recall[:-1] >= target)[0].max()])

def operating(y_true, p, thr):
    tn, fp, fn, tp = confusion_matrix(y_true, (p >= thr).astype(int), labels=[0, 1]).ravel()
    d = lambda a, b: float(a / b) if b else float("nan")
    rec, prec = d(tp, tp + fn), d(tp, tp + fp)
    return {"recall": rec, "precision": prec, "specificity": d(tn, tn + fp), "npv": d(tn, tn + fn), "f1": d(2 * prec * rec, prec + rec),
            "accuracy": d(tp + tn, len(y_true)), "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)}

def ranking(y_true, p):
    return {"pr_auc": float(average_precision_score(y_true, p)), "roc_auc": float(roc_auc_score(y_true, p)), "brier": float(brier_score_loss(y_true, p))}

oof = {}
for name, search in (("Random Forest", rf_search), ("XGBoost", xgb_search)):
    t0 = time.perf_counter()
    p_raw = cross_val_predict(clone(search.best_estimator_), X_train, y_train, cv=splits, method="predict_proba", n_jobs=1)[:, 1]
    p_cal = crossfit_isotonic(p_raw)
    thr = threshold_for_recall(y_tr, p_cal, TARGET_RECALL)
    folds = pd.DataFrame([{**ranking(y_tr[fold_id == k], p_cal[fold_id == k]), **operating(y_tr[fold_id == k], p_cal[fold_id == k], thr)}
                          for k in range(CV_FOLDS)])
    oof[name] = {"p_cal": p_cal, "threshold": thr, "folds": folds, "pooled": {**ranking(y_tr, p_cal), **operating(y_tr, p_cal, thr)}}
    print(f"{name}: OOF done in {time.perf_counter()-t0:.0f}s | threshold {thr:.4f} | pooled recall {oof[name]['pooled']['recall']:.4f}, "
          f"precision {oof[name]['pooled']['precision']:.4f}, PR-AUC {oof[name]['pooled']['pr_auc']:.4f}, ROC-AUC {oof[name]['pooled']['roc_auc']:.4f}")
pd.DataFrame({n: o["folds"][["pr_auc", "roc_auc", "brier", "recall", "precision", "specificity", "f1"]].mean() for n, o in oof.items()}).round(4)

Random Forest: OOF done in 556s | threshold 0.9961 | pooled recall 0.9805, precision 1.0000, PR-AUC 0.9992, ROC-AUC 0.9989
XGBoost: OOF done in 52s | threshold 0.9908 | pooled recall 0.9802, precision 1.0000, PR-AUC 0.9993, ROC-AUC 0.9991


,Random Forest,XGBoost
pr_auc,0.9991,0.9993
roc_auc,0.9990,0.9991
brier,0.0046,0.0044
recall,0.9805,0.9802
precision,1.0000,1.0000
specificity,1.0000,1.0000
f1,0.9901,0.9900


## 8. Model selection (protocol step 5, applied mechanically), the rule baseline and the tier analysis

In [17]:
prec = {n: o["folds"]["precision"].mean() for n, o in oof.items()}
prauc = {n: o["folds"]["pr_auc"].mean() for n, o in oof.items()}
if abs(prec["Random Forest"] - prec["XGBoost"]) >= TIE_MARGIN_PRECISION:
    WINNER = max(prec, key=prec.get); REASON = f"higher mean OOF precision at sensitivity >= {TARGET_RECALL} ({prec[WINNER]:.4f} vs {min(prec.values()):.4f})"
elif abs(prauc["Random Forest"] - prauc["XGBoost"]) >= TIE_MARGIN_PRAUC:
    WINNER = max(prauc, key=prauc.get); REASON = f"OOF precision tied; higher mean OOF PR-AUC ({prauc[WINNER]:.4f} vs {min(prauc.values()):.4f})"
else:
    WINNER = PREFERRED_MODEL_ON_TIE; REASON = (f"OOF precision and PR-AUC tied (differences < {TIE_MARGIN_PRECISION} / {TIE_MARGIN_PRAUC}); "
                                               f"{PREFERRED_MODEL_ON_TIE} by standing project decision")
THRESHOLD = oof[WINNER]["threshold"]
print("SELECTED:", WINNER, "|", REASON)
print(f"FROZEN THRESHOLD: {THRESHOLD:.6f}")
if WINNER != "XGBoost":
    raise SystemExit("Random Forest won by the protocol. The website explains each result with XGBoost's TreeSHAP, so this is a decision for "
                     "the team (keep the rule, or change the protocol before looking at any test result). Nothing was saved.")

rule_oof = ((X_train["eGFR"] < 60) | (X_train["Urine_ACR"] > 300)).astype(int).to_numpy()
print("\nOOF, training rows - plain rule vs model at the frozen threshold:")
display(pd.DataFrame({"Plain rule (no machine learning)": operating(y_tr, rule_oof, 0.5),
                      "Model at the frozen threshold": operating(y_tr, oof[WINNER]["p_cal"], THRESHOLD)}).T.round(4))

best_model = xgb_search.best_estimator_.named_steps["model"]
STAGE1_ONLY = STAGE1_FIELDS
OTHER_LABS = [c for c in INPUT_COLUMNS if c not in STAGE1_FIELDS + RENAL_MARKERS]
tiers = {}
for tier, cols in (("A: Stage 1 inputs only", STAGE1_ONLY), ("B: + other laboratory values (no eGFR / creatinine / ACR)", STAGE1_ONLY + OTHER_LABS)):
    p_raw = cross_val_predict(make_pipeline(clone(best_model), cols), X_train[cols], y_train, cv=splits, method="predict_proba", n_jobs=1)[:, 1]
    p_cal = crossfit_isotonic(p_raw); thr = threshold_for_recall(y_tr, p_cal, TARGET_RECALL)
    tiers[tier] = {"inputs": len(cols), **ranking(y_tr, p_cal), "threshold": thr, **{k: v for k, v in operating(y_tr, p_cal, thr).items() if k in ("recall", "precision", "FN", "FP")}}
tiers["C: full Stage 2 (+ eGFR, creatinine, ACR)"] = {"inputs": 48, **ranking(y_tr, oof[WINNER]["p_cal"]), "threshold": THRESHOLD,
    **{k: v for k, v in oof[WINNER]["pooled"].items() if k in ("recall", "precision", "FN", "FP")}}
display(pd.DataFrame(tiers).T.round(4))

SELECTED: XGBoost | OOF precision and PR-AUC tied (differences < 0.005 / 0.002); XGBoost by standing project decision
FROZEN THRESHOLD: 0.990769

OOF, training rows - plain rule vs model at the frozen threshold:


,recall,precision,specificity,npv,f1,accuracy,TP,FP,TN,FN
Plain rule (no machine learning),0.9826,1.0,1.0,0.9834,0.9912,0.9914,164576.0,0.0,172506.0,2918.0
Model at the frozen threshold,0.9802,1.0,1.0,0.9811,0.9900,0.9902,164173.0,8.0,172498.0,3321.0


,inputs,pr_auc,roc_auc,brier,threshold,recall,precision,FP,FN
A: Stage 1 inputs only,27.0,0.9720,0.9655,0.0636,0.0598,0.9801,0.6705,80686.0,3338.0
B: + other laboratory values (no eGFR / creatinine / ACR),45.0,0.9909,0.9905,0.0349,0.1885,0.9800,0.9031,17610.0,3345.0
"C: full Stage 2 (+ eGFR, creatinine, ACR)",48.0,0.9993,0.9991,0.0044,0.9908,0.9802,1.0000,8.0,3321.0


## 9. Fit the final calibrated pipeline, then evaluate it ONCE on the sealed test rows

In [18]:
from sklearn.calibration import CalibratedClassifierCV

final_model = CalibratedClassifierCV(clone(xgb_search.best_estimator_), method="isotonic", cv=splits, ensemble=False)
t0 = time.perf_counter(); final_model.fit(X_train, y_train)
print(f"Final model fitted in {time.perf_counter()-t0:.0f}s")
assert len(final_model.calibrated_classifiers_) == 1 and list(final_model.feature_names_in_) == INPUT_COLUMNS

# === UNSEAL THE TEST ROWS ===
X_test, y_test = X_test_sealed, y_test_sealed
y_te = y_test.to_numpy()
p_test = final_model.predict_proba(X_test)[:, list(final_model.classes_).index(HIGH_RISK_CLASS)]
test_rank, test_op = ranking(y_te, p_test), operating(y_te, p_test, THRESHOLD)
rule_test = ((X_test["eGFR"] < 60) | (X_test["Urine_ACR"] > 300)).astype(int).to_numpy()
print(f"TEST set, {len(y_te):,} rows (the same people as Stage 1's test set), threshold {THRESHOLD:.4f}")
display(pd.DataFrame({"Model": {**test_rank, **test_op}, "Plain rule": {**operating(y_te, rule_test, 0.5), "roc_auc": float(roc_auc_score(y_te, rule_test))}}).T.round(4))

Final model fitted in 70s
TEST set, 60,000 rows (the same people as Stage 1's test set), threshold 0.9908


,pr_auc,roc_auc,brier,recall,precision,specificity,npv,f1,accuracy,TP,FP,TN,FN
Model,0.9992,0.9989,0.0046,0.9805,1.0,1.0,0.9815,0.9902,0.9904,28983.0,1.0,30441.0,575.0
Plain rule,NaN,0.9915,NaN,0.9830,1.0,1.0,0.9837,0.9914,0.9916,29055.0,0.0,30442.0,503.0


## 10. Deployment: save the model, its metadata and a model card for the website

In [19]:
classes = list(final_model.classes_)
inner = final_model.calibrated_classifiers_[0].estimator
assert type(inner.named_steps["model"]).__name__ == "XGBClassifier"
pre = inner.named_steps["preprocessor"]

def imputation_defaults(preprocessor):
    out = {}
    for name, transformer, cols in preprocessor.transformers_:
        if name in ("derived_obesity", "remainder"):
            continue
        imputer = transformer if isinstance(transformer, SimpleImputer) else transformer.named_steps["impute"]
        for col, value in zip(cols, imputer.statistics_):
            out[col] = value.item() if hasattr(value, "item") else value
    return out

def categorical_answers(preprocessor):
    out = {}
    for name, transformer, cols in preprocessor.transformers_:
        if name.startswith("categorical"):
            for col, cats in zip(cols, transformer.named_steps["onehot"].categories_):
                out[col] = sorted(str(c) for c in cats)
    return out

def grouped_importance(pipeline):
    groups = []
    for n in pipeline.named_steps["preprocessor"].get_feature_names_out():
        prefix, rest = n.split("__", 1)
        if prefix == "derived_obesity":
            groups.append("Obesity")
        elif prefix in ("numeric", "binary"):
            groups.append(rest.replace("missingindicator_", "", 1))
        else:
            groups.append(max((c for c in CATEGORICAL + CATEGORICAL_BLANK_IS_CATEGORY if rest.startswith(c + "_")), key=len))
    imp = pd.Series(pipeline.named_steps["model"].feature_importances_, index=groups).groupby(level=0).sum()
    return (imp / imp.sum()).sort_values(ascending=False)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
artifact_path = MODEL_DIR / FILE_NAME
joblib.dump(final_model, artifact_path)
sha256 = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
reloaded = joblib.load(artifact_path)
assert np.array_equal(reloaded.predict_proba(X_test.iloc[:2000]), final_model.predict_proba(X_test.iloc[:2000]))

quick = bool(SAMPLE_ROWS_FOR_A_QUICK_TRIAL)
metadata = {
    "schema_version": 1, "artifact_file": FILE_NAME, "artifact_sha256": sha256, "model_name": "XGBoost",
    "model_version": f"xgboost-stage2-{dt.date.today():%Y%m%d}-{sha256[:8]}" + ("-QUICKTRIAL" if quick else ""),
    "high_risk_class": HIGH_RISK_CLASS, "threshold": THRESHOLD,
    "threshold_method": f"highest calibrated threshold with pooled out-of-fold sensitivity >= {TARGET_RECALL} "
                        f"({CV_FOLDS}-fold cross-fitted isotonic calibration, {len(y_train):,} training rows)",
    "stage": "enhanced", "features": INPUT_COLUMNS,
    "library_versions": {"python": sys.version.split()[0], **installed},
    "trained_on": data_file.name, "trained_on_sha256": data_sha256, "quick_trial": quick,
    "source": "Trained by this project's notebook ml/notebooks/CKD_Stage2_Modeling.ipynb on CKD_400k.csv, the same dataset as the Stage 1 model.",
    "rows_train_test": [len(y_train), len(y_te)], "ckd_rate_in_training_data": round(float(y_train.mean()), 4),
    "test_roc_auc": round(test_rank["roc_auc"], 4),
    "hyperparameters": xgb_summary["best_params"],
    "near_circular_warning": "eGFR, serum creatinine and urine ACR are part of how CKD is defined; in CKD_400k.csv the plain rule eGFR < 60 or "
                             "UACR > 300 agrees with the CKD label on about 99% of rows, so results are near-circular by construction",
    "output_semantics": "predict_proba(...)[:, 1] is an isotonic-calibrated probability estimated on synthetic data in which about half of "
                        "the people have CKD. Values >= threshold are classified high. This Stage 2 model uses eGFR, serum creatinine and "
                        "urine ACR, which define CKD, so it is a confirmation-level estimate, NOT independent risk evidence. Screening, not diagnosis.",
    "categorical_answers": categorical_answers(pre),
    "reference_row": imputation_defaults(pre),
}
(MODEL_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

card = {"created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "dataset": {"file": data_file.name, "sha256": data_sha256, "rows_used": len(data)},
        "split": {"train_rows": len(y_train), "test_rows": len(y_te), "same_test_rows_as_stage1": True, "random_state": RANDOM_STATE},
        "selection": {"winner": WINNER, "reason": REASON, "target_recall": TARGET_RECALL,
                      "evidence_mean_over_folds": {"precision": prec, "pr_auc": prauc},
                      "candidates": {"Random Forest": {"tuning": rf_summary, "oof_pooled": oof["Random Forest"]["pooled"], "oof_threshold": oof["Random Forest"]["threshold"]},
                                     "XGBoost": {"tuning": xgb_summary, "oof_pooled": oof["XGBoost"]["pooled"], "oof_threshold": oof["XGBoost"]["threshold"]}}},
        "threshold": {"value": THRESHOLD, "rule": metadata["threshold_method"]},
        "rule_baseline_oof": operating(y_tr, rule_oof, 0.5), "tier_ablation_oof": tiers,
        "feature_importance_builtin": grouped_importance(inner).round(6).to_dict(),
        "test_evaluation": {"ranking": test_rank, "operating_point": test_op, "rule_baseline": operating(y_te, rule_test, 0.5)}}
(MODEL_DIR / "model_card_stage2.json").write_text(json.dumps(card, indent=2, default=float), encoding="utf-8")

holdout = PROJECT_DIR / "ml" / "stage2_sealed_test.csv"
pd.concat([X_test, y_test], axis=1).to_csv(holdout, index=False)
print("Saved:", artifact_path, "|", MODEL_DIR / "model_metadata.json", "|", MODEL_DIR / "model_card_stage2.json", "|", holdout)
print(f"Model version {metadata['model_version']} | threshold {THRESHOLD:.6f} | test ROC-AUC {test_rank['roc_auc']:.4f}")
if quick:
    print("\n*** QUICK TRIAL - set SAMPLE_ROWS_FOR_A_QUICK_TRIAL = None and run the notebook again in full before using it. ***")

Saved: D:\4ADA\NEPHRON_DEVELOPMENT\Nephron\ml\model_stage2\final_model_stage2.pkl | D:\4ADA\NEPHRON_DEVELOPMENT\Nephron\ml\model_stage2\model_metadata.json | D:\4ADA\NEPHRON_DEVELOPMENT\Nephron\ml\model_stage2\model_card_stage2.json | D:\4ADA\NEPHRON_DEVELOPMENT\Nephron\ml\stage2_sealed_test.csv
Model version xgboost-stage2-20261031-e4d32208 | threshold 0.990769 | test ROC-AUC 0.9989


### Next
```
cd D:\4ADA\NEPHRON_DEVELOPMENT\SYSTEM_DEV
python verify_model_stage2.py
python model_report.py --stage 2 --csv ml\stage2_sealed_test.csv --target CKD
```
Every line of the first command must say PASS. The second writes `ml/model_stage2/model_metrics.json` for the administrator's Model page.
`ml/stage2_sealed_test.csv` is a data file - do not put it in the public repository. Then commit and push `ml/model_stage2/`
(the `.pkl`, `model_metadata.json`, `model_card_stage2.json`, `model_metrics.json`) and this notebook; the host redeploys.